# Exercise — Verify Cross-Domain Reporting Traceability

**Trailhead Provisions**' CSRD quarterly carbon disclosure spans Orders + Inventory. Run the
pipeline, trace a reported figure back to source, and find where traceability/integrity breaks.
See `INSTRUCTIONS.md`.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import GovernedCatalog

gc = GovernedCatalog("trailhead.db")
rep = gc.run_sustainability_report()
display(rep.quarterly)
rep.reconciliation

## 1. Trace one quarter to source
Pick a reported quarter, trace it to source shipments in `rep.per_order`, and isolate the rows causing the disclosed-vs-recomputed gap.

In [ ]:
q0 = rep.quarterly.iloc[0]["quarter"]
detail = rep.per_order[rep.per_order["quarter"] == q0]
suspect = detail[detail["carbon_kg"] > 100]    # the grams-not-kg slice
print(f"Quarter {q0}: {len(detail)} shipments, {len(suspect)} with implausible (grams) carbon")
rep.reconciliation[["quarter", "disclosed_carbon_kg", "recomputed_carbon_kg", "delta_kg"]]

## 2. Traceability assessment
Replace the cell below with your assessment.

### Traceability assessment

**Shared metric definition — quarterly carbon:** sum of per-shipment `carbon_kg` over all
non-cancelled orders in the quarter, **normalized to kilograms**, attributed by `order_ts`. This
must be central (MP1) so Orders and Inventory report one number.

**Where traceability holds:** every disclosed quarterly figure decomposes to per-order, then
per-shipment rows via lineage (`quarterly_sustainability` ← `order_carbon` ← `shipment` +
`orders`). Any number can be drilled to source.

**Where it breaks:** `shipment.carbon_kg` mixes units — a slice is stored in grams. The disclosed
figure is **inflated** vs the recomputed, unit-normalized figure; the reconciliation shows a large
non-zero `delta_kg` every quarter. Traceability technically holds (you can find the rows) but
**integrity** breaks at the unit boundary — the number is auditable to a wrong source value. Fix:
the MP4 downstream normalization plus a monitor (`carbon_kg > 100`) so the defect alarms before it
reaches a disclosure.